# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 박재선
- GitHub ID: lilypark0930-lab
- 작성일: 2026.10.11
- 최종 제출 URL: https://github.com/lilypark0930-lab/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape: customers 150×6, products 100×4, orders 300×5, order_items 764×5
- 결측치: 4개 데이터셋 모두 0건
- 전체 중복: 4개 데이터셋 모두 0건
- 주요 ID 중복: customer_id, product_id, order_id, order_item_id 모두 0건 (PK 유일성 확인)
- 타입 문제 후보: signup_date, order_date가 문자열(object)로 읽힘. CSV 파일에 UTF-8 BOM이 포함되어 있어 인코딩에 따라 첫 컬럼명이 깨질 수 있음

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
결측·중복·ID 중복이 모두 없는 깨끗한 데이터임. 형식상 문제는 날짜 컬럼의 타입뿐임.

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요.
날짜 타입 변환을 가장 먼저 처리해야 함. 월별 집계, 가입일과 주문일 비교 등 이후 분석이 모두 날짜 연산에 의존하기 때문임.

### 업무·분석적 의미
형식 검사를 통과했다고 데이터가 맞는것은 아님. 값 사이의 논리 관계(예: 가입일과 주문일의 선후)는 별도로 검증해야 함.

### 한계와 추가 확인 사항
사용한 데이터는 교재의 Chapter 05 전용 데이터(practice/chapter05)가 아닌 공통 샘플 데이터라, 의도적으로 넣은 오류를 다루는 실습은 하지 못함.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 컬럼명 공백 제거·소문자화, 문자열 값 앞뒤 공백 제거
- 숫자 변환 대상과 실패 건수: age, price, quantity, unit_price → 실패 0건
- 날짜 변환 대상과 실패 건수: signup_date, order_date → 실패 0건
- 결측 처리 기준: 결측이 없어 실제 대체는 없음. 기준은 PK 결측은 삭제, 수치 결측은 중앙값 대체와 표시 컬럼 추가, 날짜 변환 실패는 NaT로 두고 건수 기록으로 정함

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
모든 변환이 실패 없이 완료됨. 값 자체의 변화는 없고 타입만 바뀜.

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.
변환 실패를 조용히 결측으로 바꾸지 않고 건수를 출력하도록 함. 삭제는 다른 테이블과의 연결까지 끊고, 대체는 실제 값처럼 보이게 만들어 정보를 손실시킬 수 있음. 그래서 PK 결측만 삭제하고, 나머지는 대체하더라도 흔적을 남기는 방식을 택함.

### 업무·분석적 의미
같은 처리를 다른 데이터에 적용해도 결과를 설명할 수 있도록 처리 기준을 코드와 문서로 남기는 것이 중요함.

### 한계와 추가 확인 사항
결측이 없어서 결측 처리 기준이 실제로 잘 동작하는지는 검증하지 못함.

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 제거 0건. 보류 4쌍(8행): order_item_id만 다르고 order_id, product_id, quantity, unit_price가 같은 행 (108·109, 379·380, 531·533, 548·549)
- 표준화한 범주값: gender(F/M), payment_method(4종), order_status(3종), city(10종), category(7종) 모두 이미 일관된 표기라 변경 없음
- 허용값 밖 값: 0건
- 이상값 후보: **가입일보다 주문일이 빠른 주문 56건**. 수치 범위는 정상(age 19~69세, quantity 1~5개, price 5천~20만 원)

![중복 범주 이상값](images/step03_rules.png)


### 결과 관찰
값 범위는 모두 정상이지만, 주문 300건 중 56건(18.7%)이 고객 가입일 이전에 주문된 것으로 나타남.

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유를 작성하세요.
가입일 이전 주문은 데이터 오류일 수도 있지만, 비회원 주문 후 가입, 가입일 기록 방식 차이 등 실제 업무 사정일 수도 있음. 원인을 모르고 지우면 전체 주문의 약 19%가 사라져 분석이 크게 왜곡됨. 중복 의심 행도 같은 이유로 보류함.

### 업무·분석적 의미
"고객 가입 후 첫 구매까지 기간" 같은 지표를 계산하면 음수가 나오게 됨. 이 데이터를 그대로 쓰면 고객 생애주기 분석을 잘못 해석할 위험이 있음.

### 한계와 추가 확인 사항
샘플 데이터가 무작위로 생성되어 날짜 관계가 맞지 않는 것일 가능성이 높음. 실제 데이터라면 데이터 담당자에게 가입일 정의와 비회원 주문 정책을 확인해야 함.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건
- 생성한 파생 컬럼: order_month, order_dayofweek (orders), line_total (order_items)
- `line_total` 검증: 764행 모두 quantity × unit_price와 일치함. unit_price도 products.price와 전부 일치함

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
모든 FK가 부모 테이블에 존재하고, 파생 컬럼 계산도 정확함. 참고로 주문 이력이 없는 고객이 24명 있음.

### 나의 해석과 판단
전처리 과정에서 행을 삭제하면 다른 테이블에 연결이 끊긴 행(고아 행)이 생길 수 있음. 예를 들어 고객 행을 지우면 그 고객의 주문은 존재하지 않는 고객을 가리키게 됨. 그래서 전처리 전뿐 아니라 전처리 후에도 관계를 다시 검증해야 함.

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 1,314 | 1,314 | 삭제 대상 없음 |
| 결측 | 0 | 0 | 원본에 결측 없음 |
| 중복 | 0 | 0 | 완전 중복 없음 (중복 의심 4쌍은 보류) |
| 변환 실패 | 0 | 0 | 모든 값이 올바른 형식 |

![전처리 전후 비교](images/step05_before_after.png)


### 결과 관찰
행 수나 값 분포 변화가 의도한 결과인지 판단한 근거를 작성하세요.
행 수와 값은 변하지 않았고, 열만 3개(order_month, order_dayofweek, line_total) 늘어남.

### 나의 해석과 판단
의도한 결과임. 결측·중복·변환 실패가 모두 0건이었으므로 행이 줄지 않는 것이 맞음. 만약 행이 줄었다면 처리 로직에 문제가 있다는 신호로 봐야 함.

### 업무·분석적 의미
전처리 결과는 "얼마나 바뀌었는가"보다 "바뀐 이유를 설명할 수 있는가"가 중요함.

### 한계와 추가 확인 사항
행 수 비교만으로는 가입일 이전 주문 같은 논리 오류를 발견할 수 없음.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: data/processed/ 아래 customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md
- `python scripts/preprocess_data.py` 재실행 결과: 4개 파일과 보고서가 다시 생성되고, 터미널에 동일한 점검 결과가 출력됨
- 재실행 후 달라진 점: 없음. 같은 입력에 대해 같은 결과가 나옴

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.
노트북은 셀 실행 순서나 커널 상태에 따라 결과가 달라질 수 있음. 실제로 실습 중 커널 재시작으로 변수와 함수가 사라져 에러가 남. 스크립트는 처음부터 끝까지 같은 순서로 실행되므로, 데이터가 바뀌어도 같은 기준으로 다시 처리할 수 있음.

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본은 수정하지 않고, 처리 결과는 별도 파일로 저장한다.
2. 모든 변환과 삭제는 건수를 기록하고, 대체한 값은 표시 컬럼으로 흔적을 남긴다.
3. 형식 검사뿐 아니라 테이블 간 관계와 값 사이의 논리(날짜 선후 등)까지 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
이상값 후보의 자동 삭제임. 이번 데이터의 가입일 이전 주문처럼 전체의 약 19%가 한 번에 사라질 수 있음. 오류인지 실제 업무 사정인지 확인하지 않고 지우면 되돌리기 어려운 분석 왜곡이 생김.

### 다음 EDA에서 특히 주의할 데이터 특성
- 주문 56건이 가입일 이전 날짜임
- 주문의 약 39%가 cancelled(64건) 또는 refunded(52건)이므로 매출 분석에서는 completed만 따로 봐야 함
- 첫 달(2025-09)과 마지막 달(2026-09)은 한 달 전체가 포함되지 않았을 수 있음
- 주문 이력이 없는 고객 24명이 있음

### 현재 전처리의 한계
교재의 Chapter 05 전용 데이터 없이 깨끗한 샘플 데이터로 진행해, 결측·표기 오류·변환 실패 처리 로직이 실제로 동작하는지는 검증하지 못함. 가입일 이전 주문과 중복 의심 행은 판단을 보류한 상태임.

## 최종 제출 체크
- [0] raw 원본을 덮어쓰지 않았습니다.
- [0] 처리 전/후 비교를 남겼습니다.
- [0] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [0] clean CSV와 재실행 결과를 확인했습니다.
- [0] 개인정보/Secret이 없습니다.
- [0] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [0] 최종 Notebook 파일 URL을 제출합니다.

# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [3]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\notebooks\ch05
프로젝트 루트: c:\dev\llm-data-analysis-study\notebooks\ch05
원본 데이터 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [7]:
from pathlib import Path
RAW_DIR = Path(r"c:\dev\llm-data-analysis-study\data\raw")

In [9]:
import pandas as pd

customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")

print('원본 데이터 불러오기 완료')

원본 데이터 불러오기 완료


In [10]:

print(customers.shape)

print(products.shape)

print(orders.shape)

print(order_items.shape)


(150, 6)
(100, 4)
(300, 5)
(764, 5)


In [11]:
raw_data = {

    "customers": customers,

    "products": products,

    "orders": orders,

    "order_items": order_items,

}

for name, df in raw_data.items():

    print(name)

    print("shape:", df.shape)

    print("missing:", int(df.isna().sum().sum()))

    print("duplicated rows:", int(df.duplicated().sum()))

    print()


customers
shape: (150, 6)
missing: 0
duplicated rows: 0

products
shape: (100, 4)
missing: 0
duplicated rows: 0

orders
shape: (300, 5)
missing: 0
duplicated rows: 0

order_items
shape: (764, 5)
missing: 0
duplicated rows: 0



## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [12]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [13]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [14]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [15]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [16]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [17]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [18]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [19]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [20]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [21]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [22]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [23]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [24]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-09-15 00:00:00
주문 종료일: 2026-09-14 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-14,2026-07,Tuesday
1,2025-09-29,2025-09,Monday
2,2026-01-26,2026-01,Monday
3,2026-04-08,2026-04,Wednesday
4,2026-02-27,2026-02,Friday


In [25]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-08-26
1,2,2026-01-09
2,3,2024-08-19
3,4,2026-06-20
4,5,2024-11-20


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [26]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [27]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [28]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [29]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [32]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [33]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [34]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [35]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [36]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [37]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\notebooks\ch05\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [38]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [40]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [42]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"

In [44]:
print(ROOT)
print((ROOT / "src").exists())
print(list(ROOT.iterdir()))

c:\dev\llm-data-analysis-study
False
[WindowsPath('c:/dev/llm-data-analysis-study/.git'), WindowsPath('c:/dev/llm-data-analysis-study/.venv'), WindowsPath('c:/dev/llm-data-analysis-study/chapter01'), WindowsPath('c:/dev/llm-data-analysis-study/data'), WindowsPath('c:/dev/llm-data-analysis-study/notebooks'), WindowsPath('c:/dev/llm-data-analysis-study/README.md'), WindowsPath('c:/dev/llm-data-analysis-study/reports'), WindowsPath('c:/dev/llm-data-analysis-study/requirements.txt'), WindowsPath('c:/dev/llm-data-analysis-study/scripts')]


In [45]:
[p for p in ROOT.rglob("*.py") if ".venv" not in p.parts]

[WindowsPath('c:/dev/llm-data-analysis-study/scripts/generate_sample_data.py')]

In [51]:
import pandas as pd
from pathlib import Path

def load_sales_data(raw_dir):
    """raw 폴더의 CSV 4개를 {테이블명: DataFrame}으로 읽기"""
    raw_dir = Path(raw_dir)
    names = ["customers", "products", "orders", "order_items"]
    return {name: pd.read_csv(raw_dir / f"{name}.csv") for name in names}

def preprocess_sales_data(raw):
    """컬럼명 정리, 공백 제거, 날짜 변환, 중복 제거"""
    processed = {}
    for name, df in raw.items():
        df = df.copy()
        df.columns = df.columns.str.strip().str.lower()
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].str.strip()
        for col in df.columns:
            if "date" in col or col.endswith("_at"):
                df[col] = pd.to_datetime(df[col], errors="coerce")
        processed[name] = df.drop_duplicates().reset_index(drop=True)
    return processed

def validate_relationships(data):
    """외래키가 부모 테이블에 존재하는지 검사"""
    rules = [
        ("orders", "customer_id", "customers", "customer_id"),
        ("order_items", "order_id", "orders", "order_id"),
        ("order_items", "product_id", "products", "product_id"),
    ]
    rows = []
    for child, fk, parent, pk in rules:
        if fk in data[child].columns and pk in data[parent].columns:
            orphan = ~data[child][fk].dropna().isin(data[parent][pk])
            rows.append({"relationship": f"{child}.{fk} → {parent}.{pk}",
                         "orphan_rows": int(orphan.sum()),
                         "valid": not orphan.any()})
    return pd.DataFrame(rows)

def compare_shapes(raw, processed):
    """전처리 전후 행·열 수 비교"""
    return pd.DataFrame([
        {"table": n,
         "raw_rows": raw[n].shape[0], "raw_cols": raw[n].shape[1],
         "processed_rows": processed[n].shape[0], "processed_cols": processed[n].shape[1],
         "rows_removed": raw[n].shape[0] - processed[n].shape[0]}
        for n in raw
        ])

In [52]:
module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\3760341559.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\kidol\AppData\Local\Temp\ipykernel_20264\3760341559.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

,table,raw_rows,raw_cols,processed_rows,processed_cols,rows_removed
0,customers,150,6,150,6,0
1,products,100,4,100,4,0
2,orders,300,5,300,5,0
3,order_items,764,5,764,5,0


In [54]:
def duplicate_summary(data, key_columns=None):
    """테이블별 완전 중복 행 수와 기본키 중복 수를 정리"""
    key_columns = key_columns or {}
    rows = []
    for name, df in data.items():
        key = key_columns.get(name)
        row = {
            "table": name,
            "rows": len(df),
            "duplicate_rows": int(df.duplicated().sum()),
        }
        if key and key in df.columns:
            row["key_column"] = key
            row["duplicate_keys"] = int(df[key].duplicated().sum())
        rows.append(row)
    return pd.DataFrame(rows)

In [55]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,table,rows,duplicate_rows,key_column,duplicate_keys
0,customers,150,0,customer_id,0
1,products,100,0,product_id,0
2,orders,300,0,order_id,0
3,order_items,764,0,order_item_id,0


In [57]:
module_relationship_checks


,relationship,orphan_rows,valid
0,orders.customer_id → customers.customer_id,0,True
1,order_items.order_id → orders.order_id,0,True
2,order_items.product_id → products.product_id,0,True


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


실행 완료 : 이미지 파일 첨부

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```
1. 원본 데이터를 직접 수정하는 문제: 있음

customers = customers.drop_duplicates()처럼 같은 변수에 덮어쓰면 원본이 사라짐. 결과가 이상해도 전처리 전과 비교하거나 되돌릴 수 없음. 원본은 그대로 두고 customers_clean = customers.copy()처럼 별도 변수에서 작업하는 것이 안전함.

2. 결측치 처리 방식: 개선 필요

평균은 이상값에 약함. age에 999나 -1 같은 잘못된 값이 섞여 있으면 평균이 왜곡되므로, 범위를 먼저 검증하고 평균 대신 중앙값을 쓰는 것이 안전함. 또한 중복 제거 전에 평균을 구하면 중복 고객이 두 번 반영되므로 중복 제거를 먼저 해야 함. 채운 값과 원래 값을 구분할 수 있도록 age_missing 같은 표시 컬럼을 남기는 것이 좋음. age가 문자열로 읽혔다면 mean() 자체가 에러를 냄.

3. 날짜 변환 실패 확인: 없음

pd.to_datetime은 기본 설정에서 이상한 값이 하나라도 있으면 전체가 에러로 멈춤. 반대로 errors="coerce"만 붙이면 실패한 값이 조용히 NaT가 되어 문제를 놓치게 됨. 변환 후 실패 건수를 반드시 세어 봐야 함.

4. 중복 제거 기준: 불충분

drop_duplicates()는 모든 컬럼 값이 완전히 같은 행만 제거함. 같은 customer_id인데 다른 컬럼 값이 약간 다른 행이나, "Kim "과 "Kim"처럼 공백·대소문자만 다른 행은 잡지 못함. 기본키(customer_id) 기준 중복도 따로 확인하고, 제거 건수를 기록해야 함.

5. line_total 계산 전 숫자형 변환: 반드시 필요

가장 위험한 부분임. quantity가 숫자이고 unit_price가 "1000" 같은 문자열이면, 에러 없이 문자열이 반복되어 3 * "1000"이 "100010001000"이 됨. 결과가 그럴듯해 보여 발견하기 어려움. "1,000"처럼 쉼표가 포함된 값도 변환이 필요함.

6. 더 안전한 코드로 수정

##
import pandas as pd

# 0) 원본 보존
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

# 1) 중복 제거를 먼저: 완전 중복 제거 + 기본키 중복 확인
before = len(customers_clean)
customers_clean = customers_clean.drop_duplicates()
print("완전 중복 제거:", before - len(customers_clean), "건")
print("customer_id 중복:", customers_clean["customer_id"].duplicated().sum(), "건")

# 2) age: 숫자 변환 → 범위 검증 → 결측 표시 → 중앙값 대체
customers_clean["age"] = pd.to_numeric(customers_clean["age"], errors="coerce")
customers_clean["age"] = customers_clean["age"].where(customers_clean["age"].between(0, 120))
customers_clean["age_missing"] = customers_clean["age"].isna()
customers_clean["age"] = customers_clean["age"].fillna(customers_clean["age"].median())

# 3) 날짜 변환 + 실패 건수 확인
raw_date = orders_clean["order_date"]
orders_clean["order_date"] = pd.to_datetime(raw_date, errors="coerce")
failed = orders_clean["order_date"].isna() & raw_date.notna()
print("날짜 변환 실패:", failed.sum(), "건", raw_date[failed].unique()[:5])

# 4) 숫자 변환 후 line_total 계산
for col in ["quantity", "unit_price"]:
    raw = order_items_clean[col]
    order_items_clean[col] = pd.to_numeric(
        raw.astype(str).str.replace(",", "").str.strip(), errors="coerce"
    )
    print(f"{col} 변환 실패:", (order_items_clean[col].isna() & raw.notna()).sum(), "건")

print("수량 0 이하:", (order_items_clean["quantity"] <= 0).sum(), "건")
print("단가 음수:", (order_items_clean["unit_price"] < 0).sum(), "건")

order_items_clean["line_total"] = order_items_clean["quantity"] * order_items_clean["unit_price"]
##

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [4]:

import pandas as pd
from pathlib import Path

# 프로젝트 루트 찾기
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

# 전처리 후 파일 불러오기
names = ["customers", "products", "orders", "order_items"]
clean = {n: pd.read_csv(ROOT / "data" / "processed" / f"{n}_clean.csv") for n in names}

customers_clean = clean["customers"]
products_clean = clean["products"]
orders_clean = clean["orders"]
order_items_clean = clean["order_items"]

print(ROOT)
{n: df.shape for n, df in clean.items()}

c:\dev\llm-data-analysis-study


{'customers': (150, 6),
 'products': (100, 4),
 'orders': (300, 7),
 'order_items': (764, 6)}

In [6]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.
def missing_summary(df):
    """컬럼별 결측치 개수와 비율"""
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_ratio": df.isna().mean().round(4),
    })

datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
}

missing_frames = []
for name, df in datasets.items():
    summary = missing_summary(df).reset_index()            # 인덱스(컬럼명)를 일반 컬럼으로
    summary = summary.rename(columns={"index": "column"})   # 'index' → 'column'
    summary.insert(0, "dataset", name)                     # 맨 앞에 데이터셋 이름 추가
    missing_frames.append(summary)

missing_all = pd.concat(missing_frames, ignore_index=True)
missing_all

,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [7]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_outliers = customers_clean[(customers_clean["age"] < 18) | (customers_clean["age"] > 100)]
print(len(age_outliers), "건")
customers_clean["age"].describe()

0 건


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [8]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_orders = orders_clean.groupby("order_month").size().rename("order_count")
monthly_orders

order_month
2025-09     9
2025-10    27
2025-11    22
2025-12    33
2026-01    29
2026-02    17
2026-03    35
2026-04    27
2026-05    28
2026-06    25
2026-07    24
2026-08    16
2026-09     8
Name: order_count, dtype: int64

In [9]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
order_items_clean.sort_values("line_total", ascending=False).head(10)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
493,494,190,70,5,198000,990000
78,79,35,70,5,198000,990000
631,632,247,58,5,197000,985000
604,605,235,58,5,197000,985000
113,114,51,58,5,197000,985000
569,570,221,58,5,197000,985000
45,46,21,58,5,197000,985000


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
